In [ ]:
import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import Delaunay, distance_matrix
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')


# Data configuration
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

# Target genes for AAD group
AAD_TARGET_GENES = [
    'Eno1', 'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm'
]

In [8]:
class SpatialGNN(nn.Module):
    """GNN for learning spatial patterns"""
    
    def __init__(self, input_dim: int, hidden_dim: int = 128, 
                 embedding_dim: int = 64, num_layers: int = 3):
        super(SpatialGNN, self).__init__()
        
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # First layer
        self.convs.append(GATConv(input_dim, hidden_dim, heads=4, concat=True))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim * 4))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * 4, hidden_dim, heads=4, concat=True))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * 4))
        
        # Output layer
        self.convs.append(GATConv(hidden_dim * 4, embedding_dim, heads=1, concat=False))
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x, edge_index):
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, edge_index)
            x = self.batch_norms[i](x)
            x = F.elu(x)
            x = self.dropout(x)
        x = self.convs[-1](x, edge_index)
        return x

In [9]:

class GeneToMzMatcher:
    """Match gene expression patterns to m/z intensity patterns"""
    
    def __init__(self, output_dir: str = './gene_to_mz_results',
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'gene_visualizations'), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.groups = ['YC', 'YAD', 'AC', 'AAD']
        
    def load_all_data(self):
        """Load all RNA and MSI data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            self.rna_data[sample_id] = sc.read_h5ad(path)
            print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            self.msi_data[sample_id] = sc.read_h5ad(path)
            print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def check_gene_availability(self):
        """Check which target genes are available in the data"""
        print("\nChecking gene availability across samples...")
        gene_availability = {}
        
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if gene in self.rna_data[sample_id].var_names:
                    gene_availability[gene][sample_id] = True
                else:
                    gene_availability[gene][sample_id] = False
        
        # Summary
        for gene in AAD_TARGET_GENES:
            available_count = sum(gene_availability[gene].values())
            print(f"  {gene}: {available_count}/16 samples")
        
        return gene_availability
    
    def apply_spatial_smoothing(self, coords, values, sigma: float = 100.0):
        """Apply Gaussian spatial smoothing"""
        smoothed = np.zeros_like(values)
        
        for i in range(len(coords)):
            dists = np.sqrt(np.sum((coords - coords[i])**2, axis=1))
            weights = np.exp(-dists**2 / (2 * sigma**2))
            weights /= weights.sum()
            smoothed[i] = values.T @ weights if values.ndim > 1 else values @ weights
        
        return smoothed
    
    def build_spatial_graph(self, coords) -> Tuple[torch.Tensor, torch.Tensor]:
        """Build spatial graph from coordinates"""
        try:
            tri = Delaunay(coords)
            edges = set()
            for simplex in tri.simplices:
                for i in range(len(simplex)):
                    for j in range(i + 1, len(simplex)):
                        edges.add(tuple(sorted([simplex[i], simplex[j]])))
            edge_list = list(edges)
        except:
            # Fallback to k-NN
            dist_mat = distance_matrix(coords, coords)
            edge_list = []
            k = 6
            for i in range(len(coords)):
                nearest = np.argsort(dist_mat[i])[1:k+1]
                for j in nearest:
                    edge_list.append([i, j])
        
        # Bidirectional edges
        edge_index = []
        for edge in edge_list:
            edge_index.append([edge[0], edge[1]])
            edge_index.append([edge[1], edge[0]])
        
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        coords_tensor = torch.tensor(coords, dtype=torch.float32)
        
        return edge_index, coords_tensor
    
    def train_spatial_gnn(self, data_dict: Dict, input_dim: int, epochs: int = 100):
        """Train GNN to learn spatial patterns"""
        model = SpatialGNN(input_dim=input_dim).to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
        
        model.train()
        for epoch in range(epochs):
            total_loss = 0
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                optimizer.zero_grad()
                
                embeddings = model(data.x, data.edge_index)
                
                # Spatial consistency loss
                spatial_dist = torch.cdist(data.pos, data.pos)
                embedding_dist = torch.cdist(embeddings, embeddings)
                
                spatial_dist = spatial_dist / (spatial_dist.max() + 1e-8)
                embedding_dist = embedding_dist / (embedding_dist.max() + 1e-8)
                
                loss = F.mse_loss(embedding_dist, spatial_dist)
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(data_dict):.4f}")
        
        return model
    
    def extract_gene_spatial_pattern(self, sample_id: str, gene: str, 
                                     smooth_sigma: float = 100.0):
        """Extract spatial pattern for a specific gene"""
        adata = self.rna_data[sample_id]
        
        if gene not in adata.var_names:
            return None, None
        
        coords = np.column_stack([adata.obs['x_um'].values, 
                                  adata.obs['y_um'].values])
        
        gene_idx = list(adata.var_names).index(gene)
        
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        # Apply spatial smoothing
        gene_expr_smooth = self.apply_spatial_smoothing(coords, gene_expr, smooth_sigma)
        
        # Normalize
        gene_expr_smooth = (gene_expr_smooth - gene_expr_smooth.min()) / (gene_expr_smooth.max() - gene_expr_smooth.min() + 1e-8)
        
        return coords, gene_expr_smooth
    
    def extract_mz_spatial_patterns(self, sample_id: str, smooth_sigma: float = 100.0):
        """Extract spatial patterns for all m/z features"""
        adata = self.msi_data[sample_id]
        
        coords = np.column_stack([adata.obs['x_um'].values, 
                                  adata.obs['y_um'].values])
        
        if hasattr(adata.X, 'toarray'):
            mz_intensities = adata.X.toarray()
        else:
            mz_intensities = adata.X.copy()
        
        # Apply spatial smoothing to each m/z
        mz_smooth = np.zeros_like(mz_intensities)
        for i in range(mz_intensities.shape[1]):
            mz_smooth[:, i] = self.apply_spatial_smoothing(
                coords, mz_intensities[:, i], smooth_sigma
            )
        
        # Normalize each m/z
        for i in range(mz_smooth.shape[1]):
            mz_smooth[:, i] = (mz_smooth[:, i] - mz_smooth[:, i].min()) / (mz_smooth[:, i].max() - mz_smooth[:, i].min() + 1e-8)
        
        return coords, mz_smooth
    
    def interpolate_patterns_to_common_grid(self, coords1, pattern1, coords2, pattern2, 
                                            grid_resolution: int = 100):
        """Interpolate both patterns to a common grid for comparison"""
        from scipy.interpolate import griddata
        
        # Define common grid
        x_min = min(coords1[:, 0].min(), coords2[:, 0].min())
        x_max = max(coords1[:, 0].max(), coords2[:, 0].max())
        y_min = min(coords1[:, 1].min(), coords2[:, 1].min())
        y_max = max(coords1[:, 1].max(), coords2[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_resolution),
            np.linspace(y_min, y_max, grid_resolution)
        )
        
        # Interpolate both patterns
        pattern1_grid = griddata(coords1, pattern1, (grid_x, grid_y), method='cubic', fill_value=0)
        pattern2_grid = griddata(coords2, pattern2, (grid_x, grid_y), method='cubic', fill_value=0)
        
        # Flatten for correlation
        mask = (~np.isnan(pattern1_grid)) & (~np.isnan(pattern2_grid))
        pattern1_flat = pattern1_grid[mask]
        pattern2_flat = pattern2_grid[mask]
        
        return pattern1_flat, pattern2_flat, grid_x, grid_y, pattern1_grid, pattern2_grid
    
    def compute_spatial_similarity(self, coords1, pattern1, coords2, pattern2):
        """Compute multiple spatial similarity metrics"""
        # Interpolate to common grid
        p1_flat, p2_flat, _, _, _, _ = self.interpolate_patterns_to_common_grid(
            coords1, pattern1, coords2, pattern2
        )
        
        if len(p1_flat) < 10:  # Not enough overlap
            return {'pearson': 0, 'spearman': 0, 'cosine': 0}
        
        # Pearson correlation
        pearson_corr, _ = pearsonr(p1_flat, p2_flat)
        
        # Spearman correlation
        spearman_corr, _ = spearmanr(p1_flat, p2_flat)
        
        # Cosine similarity
        cosine_sim = np.dot(p1_flat, p2_flat) / (np.linalg.norm(p1_flat) * np.linalg.norm(p2_flat) + 1e-8)
        
        return {
            'pearson': pearson_corr,
            'spearman': spearman_corr,
            'cosine': cosine_sim
        }
    
    def find_matching_mz_features(self, gene: str, rna_sample_id: str, 
                                  msi_sample_ids: List[str], top_k: int = 20,
                                  smooth_sigma: float = 100.0):
        """Find m/z features matching a gene's spatial pattern"""
        print(f"\nFinding m/z matches for {gene} in {rna_sample_id}...")
        
        # Extract gene spatial pattern
        gene_coords, gene_pattern = self.extract_gene_spatial_pattern(
            rna_sample_id, gene, smooth_sigma
        )
        
        if gene_pattern is None:
            print(f"  Gene {gene} not found in {rna_sample_id}")
            return None
        
        all_matches = []
        
        for msi_sample_id in msi_sample_ids:
            print(f"  Comparing with MSI sample {msi_sample_id}...")
            
            # Extract m/z patterns
            mz_coords, mz_patterns = self.extract_mz_spatial_patterns(
                msi_sample_id, smooth_sigma
            )
            
            mz_features = self.msi_data[msi_sample_id].var_names
            
            # Compute similarity for each m/z
            for mz_idx in range(mz_patterns.shape[1]):
                mz_pattern = mz_patterns[:, mz_idx]
                
                similarities = self.compute_spatial_similarity(
                    gene_coords, gene_pattern, mz_coords, mz_pattern
                )
                
                all_matches.append({
                    'gene': gene,
                    'rna_sample': rna_sample_id,
                    'msi_sample': msi_sample_id,
                    'mz_feature': mz_features[mz_idx],
                    'mz_index': mz_idx,
                    'pearson': similarities['pearson'],
                    'spearman': similarities['spearman'],
                    'cosine': similarities['cosine'],
                    'average_score': np.mean([similarities['pearson'], 
                                             similarities['spearman'], 
                                             similarities['cosine']])
                })
        
        # Convert to DataFrame and sort
        matches_df = pd.DataFrame(all_matches)
        matches_df = matches_df.sort_values('average_score', ascending=False)
        
        print(f"\n  Top match: {matches_df.iloc[0]['mz_feature']} "
              f"(MSI: {matches_df.iloc[0]['msi_sample']}, "
              f"Score: {matches_df.iloc[0]['average_score']:.3f})")
        
        return matches_df.head(top_k)
    
    def visualize_gene_mz_match(self, gene: str, rna_sample_id: str, 
                                mz_feature: str, msi_sample_id: str,
                                smooth_sigma: float = 100.0):
        """Visualize gene and matching m/z spatial patterns"""
        # Extract patterns
        gene_coords, gene_pattern = self.extract_gene_spatial_pattern(
            rna_sample_id, gene, smooth_sigma
        )
        
        mz_coords, mz_patterns = self.extract_mz_spatial_patterns(
            msi_sample_id, smooth_sigma
        )
        
        mz_idx = list(self.msi_data[msi_sample_id].var_names).index(mz_feature)
        mz_pattern = mz_patterns[:, mz_idx]
        
        # Compute similarity
        similarities = self.compute_spatial_similarity(
            gene_coords, gene_pattern, mz_coords, mz_pattern
        )
        
        # Interpolate to common grid
        p1_flat, p2_flat, grid_x, grid_y, gene_grid, mz_grid = \
            self.interpolate_patterns_to_common_grid(
                gene_coords, gene_pattern, mz_coords, mz_pattern
            )
        
        # Create visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Gene pattern
        im1 = axes[0, 0].scatter(gene_coords[:, 0], gene_coords[:, 1], 
                                 c=gene_pattern, cmap='viridis', s=50)
        axes[0, 0].set_title(f'{gene} Expression\n{rna_sample_id}', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # m/z pattern
        im2 = axes[0, 1].scatter(mz_coords[:, 0], mz_coords[:, 1], 
                                 c=mz_pattern, cmap='viridis', s=20)
        axes[0, 1].set_title(f'm/z {mz_feature} Intensity\n{msi_sample_id}', fontweight='bold')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Interpolated gene
        im3 = axes[1, 0].imshow(gene_grid, origin='lower', cmap='viridis', aspect='auto')
        axes[1, 0].set_title('Gene (Interpolated)', fontweight='bold')
        plt.colorbar(im3, ax=axes[1, 0])
        
        # Interpolated m/z
        im4 = axes[1, 1].imshow(mz_grid, origin='lower', cmap='viridis', aspect='auto')
        axes[1, 1].set_title('m/z (Interpolated)', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 1])
        
        # Scatter plot
        axes[0, 2].scatter(p1_flat, p2_flat, alpha=0.5, s=10)
        axes[0, 2].set_xlabel('Gene Expression (normalized)')
        axes[0, 2].set_ylabel('m/z Intensity (normalized)')
        axes[0, 2].set_title(f'Correlation Plot\nPearson: {similarities["pearson"]:.3f}', 
                            fontweight='bold')
        
        # Similarity metrics
        metrics_text = f"""Similarity Metrics:
        
Pearson:  {similarities['pearson']:.3f}
Spearman: {similarities['spearman']:.3f}
Cosine:   {similarities['cosine']:.3f}
Average:  {np.mean(list(similarities.values())):.3f}

RNA Sample: {rna_sample_id}
MSI Sample: {msi_sample_id}
Gene: {gene}
m/z: {mz_feature}"""
        
        axes[1, 2].text(0.1, 0.5, metrics_text, fontsize=11, 
                       verticalalignment='center', family='monospace')
        axes[1, 2].axis('off')
        
        plt.suptitle(f'Spatial Pattern Match: {gene} ↔ m/z {mz_feature}', 
                    fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'{gene}_{rna_sample_id}_mz{mz_feature}_{msi_sample_id}.png'
        filepath = os.path.join(self.output_dir, 'gene_visualizations', filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        
        return filepath
    
    def analyze_all_target_genes(self, smooth_sigma: float = 100.0, top_k: int = 20):
        """Analyze all AAD target genes across all samples"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z SPATIAL PATTERN MATCHING")
        print("="*70)
        
        gene_availability = self.check_gene_availability()
        
        all_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*70}")
            print(f"Analyzing gene: {gene}")
            print(f"{'='*70}")
            
            # Find samples where gene is available
            available_samples = [s for s, avail in gene_availability[gene].items() if avail]
            
            if not available_samples:
                print(f"  {gene} not found in any samples, skipping...")
                continue
            
            # Analyze each RNA sample against all MSI samples
            for rna_sample in available_samples:
                matches_df = self.find_matching_mz_features(
                    gene, rna_sample, MSI_SAMPLE_IDS, top_k, smooth_sigma
                )
                
                if matches_df is not None:
                    all_results.append(matches_df)
                    
                    # Visualize top 3 matches
                    for i in range(min(3, len(matches_df))):
                        match = matches_df.iloc[i]
                        print(f"\n  Visualizing match {i+1}: {match['mz_feature']} "
                              f"in {match['msi_sample']} (score: {match['average_score']:.3f})")
                        
                        self.visualize_gene_mz_match(
                            gene, rna_sample, 
                            match['mz_feature'], match['msi_sample'],
                            smooth_sigma
                        )
        
        # Combine all results
        if all_results:
            combined_results = pd.concat(all_results, ignore_index=True)
            combined_results = combined_results.sort_values('average_score', ascending=False)
            
            # Save results
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_all.csv')
            combined_results.to_csv(output_file, index=False)
            print(f"\n\nAll results saved to: {output_file}")
            
            # Summary statistics
            print("\n" + "="*70)
            print("SUMMARY STATISTICS")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gene_results = combined_results[combined_results['gene'] == gene]
                if len(gene_results) > 0:
                    best_match = gene_results.iloc[0]
                    print(f"\n{gene}:")
                    print(f"  Best m/z: {best_match['mz_feature']}")
                    print(f"  MSI sample: {best_match['msi_sample']}")
                    print(f"  RNA sample: {best_match['rna_sample']}")
                    print(f"  Score: {best_match['average_score']:.3f}")
                    print(f"  Total matches analyzed: {len(gene_results)}")
            
            return combined_results
        
        return None

In [10]:
def main():
    """Main analysis pipeline"""
    print("="*70)
    print("GENE-TO-M/Z SPATIAL PATTERN MATCHING PIPELINE")
    print("Identifying MSI features matching gene expression patterns")
    print("="*70)
    
    # Initialize matcher
    matcher = GeneToMzMatcher(output_dir='./gene_to_mz_results')
    
    # Load data
    matcher.load_all_data()
    
    # Run analysis
    results = matcher.analyze_all_target_genes(
        smooth_sigma=100.0,  # Adjust based on your data
        top_k=20  # Top 20 matches per gene per sample
    )
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)
    print(f"Results directory: {matcher.output_dir}/")
    print(f"  - gene_to_mz_matches_all.csv: All matches")
    print(f"  - gene_visualizations/: Individual gene-m/z match visualizations")
    
    return matcher, results

In [11]:
if __name__ == "__main__":
    matcher, results = main()

GENE-TO-M/Z SPATIAL PATTERN MATCHING PIPELINE
Identifying MSI features matching gene expression patterns
Loading RNA-seq data...
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z SPATIAL PATTERN MATCHING

Checking gene availability across samples...
  Mapt: 16/16 samples
  Thy1: 16/16 samples
  Pmch: 16/16 samples
  Atp1a3: 16/16 samples
  Rac1: 16/16 sample

KeyboardInterrupt: 